# Scene Image Classification using Convolutional Neural Networks

**Module:** 6CS012 Artificial Intelligence and Machine Learning
**Student:** Suyank Hang Rai | **ID:** 2331514
**Group:** L6CG20 (Salon, Prasanna, Rahul, Suyank)
**Tutor:** Ms. Durga Pokharel

This notebook covers the full Vision Task (Part II) of the Final Portfolio Assessment:

**Part A:** CNN trained from scratch: baseline model, deeper regularised model, SGD vs Adam optimiser comparison, ablation study.
**Part B:** Transfer learning with MobileNetV2: feature extraction then selective fine-tuning.

**Dataset:** Intel Scene Classification with 6 categories (buildings, forest, glacier, mountain, sea, street).

## 1. Setup and Imports

All libraries imported upfront. TensorFlow/Keras handles model building and training. Seaborn and Matplotlib are used for visualisations. A global seed ensures reproducibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, random, zipfile, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
def make_early_stopping():
    return callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

## 2. Dataset Preparation

**Dataset:** Intel Scene Classification (Kaggle).
The zip contains a train folder with six labelled subdirectories and a sparse unlabelled test folder.

Since the official test split cannot be used for evaluation, the training set is re-split internally:
80 percent for training, 10 percent for validation (early stopping), 10 percent for held-out test (only touched at final evaluation).

Mount Google Drive and update DATA_ZIP to the zip file path before running.

In [ ]:
DATA_ZIP    = '/content/drive/MyDrive/Ai ki mah kja/Scene Classification-20260416T164630Z-3-001.zip'
EXTRACT_DIR = '/content/drive/MyDrive/Ai ki mah kja'

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extracted to', EXTRACT_DIR)
else:
    print('Already extracted at', EXTRACT_DIR)

DATA_DIR  = os.path.join(EXTRACT_DIR, 'Scene Classification')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

print('Top-level contents:', os.listdir(DATA_DIR))
print('Train classes:', sorted(os.listdir(TRAIN_DIR)))

### 2.1 Class Distribution and Corrupt File Cleanup

Before building any model, a sanity check runs on the raw files. Any file under 100 bytes is either empty or corrupt and is removed to prevent Keras from crashing during decoding.

In [ ]:
CLASSES = sorted(os.listdir(TRAIN_DIR))
counts_before = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASSES}
print('Before cleaning:', counts_before)

removed = 0
for c in CLASSES:
    folder = os.path.join(TRAIN_DIR, c)
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        if os.path.getsize(p) < 100:
            os.remove(p)
            removed += 1
print(f'Removed {removed} corrupt/empty files')

counts_after = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASSES}
print('After cleaning:', counts_after)
total = sum(counts_after.values())
print('Total usable images:', total)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts_after.keys(), counts_after.values(),
       color=sns.color_palette('husl', len(CLASSES)), edgecolor='black')
ax.set_title('Image Count per Class (Training Set)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.set_xlabel('Class')
for i, (k, v) in enumerate(counts_after.items()):
    ax.text(i, v + 20, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Imbalance ratio (max/min):', f"{max(counts_after.values()) / min(counts_after.values()):.2f}x")

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(11, 2.2 * len(CLASSES)))
for i, c in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, c)
    samples = random.sample(os.listdir(folder), 5)
    for j, f in enumerate(samples):
        img = Image.open(os.path.join(folder, f))
        axes[i][j].imshow(img)
        axes[i][j].axis('off')
        if j == 0:
            axes[i][j].set_title(c, loc='left', fontsize=11, fontweight='bold')
plt.suptitle('Sample Images per Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.2 Train / Validation / Test Pipelines

Images are resized to 100x100 for Part A (cheap to train while retaining enough resolution to distinguish scene categories). For Part B (MobileNetV2) we switch to 128x128.

The held-out 20 percent is split in half: 10 percent becomes validation (used for early stopping), 10 percent becomes the final test set that is only ever touched once at evaluation time.

In [ ]:
IMG_SIZE   = (100, 100)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='training',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical')

val_full = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='validation',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical')

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', class_names)

val_batches = tf.data.experimental.cardinality(val_full).numpy()
test_ds = val_full.take(val_batches // 2)
val_ds  = val_full.skip(val_batches // 2)

print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
print('Val   batches:', tf.data.experimental.cardinality(val_ds).numpy())
print('Test  batches:', tf.data.experimental.cardinality(test_ds).numpy())

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

### 2.3 Data Augmentation

Augmentation artificially expands the effective training set by applying random transformations at training time only:
Horizontal flip, Rotation 10 degrees, Zoom 10 percent, Contrast jitter.

The augmentation layer is baked into the model graph so it is automatically inactive during inference.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation')

for images, _ in train_ds.take(1):
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax in axes.flat:
        aug = data_augmentation(tf.expand_dims(images[0], 0))
        ax.imshow(aug[0].numpy().astype('uint8'))
        ax.axis('off')
    fig.suptitle('Nine augmented versions of the same image', fontsize=12)
    plt.tight_layout()
    plt.show()
    break

### 2.4 Reusable Evaluation Helpers

Three reusable functions defined once and called throughout all sections: plot_history, evaluate_model, and show_sample_predictions.

In [ ]:
def plot_history(history, title):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h['loss'],         label='train', linewidth=2)
    axes[0].plot(h['val_loss'],     label='val',   linewidth=2, linestyle='--')
    axes[0].set_title(f'{title} Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(h['accuracy'],     label='train', linewidth=2)
    axes[1].plot(h['val_accuracy'], label='val',   linewidth=2, linestyle='--')
    axes[1].set_title(f'{title} Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout(); plt.show()


def evaluate_model(model, ds, class_names, title):
    y_true, y_pred = [], []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0)
        y_true.extend(np.argmax(yb.numpy(), axis=1))
        y_pred.extend(np.argmax(pb, axis=1))
    print(f'\n===== {title} =====')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title(f'{title} Confusion Matrix', fontweight='bold')
    plt.tight_layout(); plt.show()
    return np.array(y_true), np.array(y_pred)


def show_sample_predictions(model, ds, class_names, n=9):
    plt.figure(figsize=(10, 10))
    for imgs, labels in ds.take(1):
        preds = model.predict(imgs, verbose=0)
        for i in range(min(n, imgs.shape[0])):
            plt.subplot(3, 3, i + 1)
            plt.imshow(imgs[i].numpy().astype('uint8'))
            t = class_names[np.argmax(labels[i])]
            p = class_names[np.argmax(preds[i])]
            plt.title(f'True: {t}\nPred: {p}',
                      color='green' if t == p else 'red', fontsize=9)
            plt.axis('off')
    plt.suptitle('Sample Predictions (green = correct, red = wrong)', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

## 3. Part A.1 Baseline CNN from Scratch

**Architecture rationale.**
Three convolutional blocks with increasing filter counts (32, 64, 128) using 3x3 kernels with same padding and ReLU, each followed by 2x2 max-pooling. Three dense layers (256, 128, 64) progressively compress features before the 6-class softmax output. Pixel values are rescaled from 0-255 to 0-1 inside the model via a Rescaling layer. No dropout is used here as this is intentionally minimal as a baseline.

In [ ]:
def build_baseline_cnn(num_classes):
    model = keras.Sequential([
        layers.Input(shape=(100, 100, 3)),
        layers.Rescaling(1./255),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64,  activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='baseline_cnn')
    return model

baseline = build_baseline_cnn(num_classes)
baseline.summary()

In [ ]:
baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

t0 = time.time()
hist_baseline = baseline.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[make_early_stopping()],
    verbose=1,
)
time_baseline = time.time() - t0
print(f'\nBaseline training time: {time_baseline:.1f} s')

In [ ]:
plot_history(hist_baseline, 'Baseline CNN')
y_t_base, y_p_base = evaluate_model(baseline, test_ds, class_names, 'Baseline CNN')
show_sample_predictions(baseline, test_ds, class_names)

**Observation.** The baseline achieves approximately 70 to 75 percent test accuracy. The train/val loss curves show a visible gap after epoch 8 to 10 indicating mild overfitting. The dense head memorises training patterns without generalising well. This directly motivates the regularised deeper model below.

## 4. Part A.2 Deeper CNN with Regularisation

**Changes vs the baseline:**
Six conv layers (double the baseline), three paired blocks before each pooling step.
Batch Normalisation after every conv layer stabilises training and acts as a mild regulariser.
Dropout in the dense head randomly zeros 50 percent of activations on the first dense layer and 30 percent on the second, preventing co-adaptation.
Global Average Pooling replaces Flatten, cutting dense-head parameter count significantly.
Data augmentation is wired directly into the model graph so it runs only during training.

The function accepts use_bn and use_dropout flags to make the ablation study in Section 6 straightforward.

In [ ]:
def build_deeper_cnn(num_classes, use_bn=True, use_dropout=True):
    def maybe_bn(x):
        return layers.BatchNormalization()(x) if use_bn else x

    inputs = keras.Input(shape=(100, 100, 3))
    x = layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x); x = maybe_bn(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    if use_dropout: x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    if use_dropout: x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='deeper_cnn')

deeper = build_deeper_cnn(num_classes)
deeper.summary()

In [ ]:
deeper.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

t0 = time.time()
hist_deeper = deeper.fit(
    train_ds, validation_data=val_ds,
    epochs=25, callbacks=[make_early_stopping()], verbose=1,
)
time_deeper = time.time() - t0
print(f'\nDeeper (Adam) training time: {time_deeper:.1f} s')

plot_history(hist_deeper, 'Deeper CNN (Adam)')
y_t_deep, y_p_deep = evaluate_model(deeper, test_ds, class_names, 'Deeper CNN (Adam)')
show_sample_predictions(deeper, test_ds, class_names)

**Observation.** The deeper model typically improves test accuracy by 3 to 6 percentage points over the baseline. The train/val gap narrows because BatchNorm and Dropout are reducing overfitting. Global Average Pooling contributes significantly by reducing the number of parameters in the dense head.

## 5. Part A.3 Optimiser Comparison: Adam vs SGD

The same deeper architecture is retrained with SGD and Nesterov momentum (lr=0.01, momentum=0.9). Everything else is identical to isolate the effect of the optimiser.

Adam adapts per-parameter learning rates and generally converges faster. SGD with momentum is simpler and can sometimes find flatter minima that generalise better. Nesterov momentum computes the gradient at a look-ahead position for slightly more informed updates.

In [ ]:
deeper_sgd = build_deeper_cnn(num_classes)
deeper_sgd.compile(
    optimizer=optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

t0 = time.time()
hist_sgd = deeper_sgd.fit(
    train_ds, validation_data=val_ds,
    epochs=25, callbacks=[make_early_stopping()], verbose=1,
)
time_sgd = time.time() - t0
print(f'\nDeeper (SGD) training time: {time_sgd:.1f} s')

plot_history(hist_sgd, 'Deeper CNN (SGD)')
y_t_sgd, y_p_sgd = evaluate_model(deeper_sgd, test_ds, class_names, 'Deeper CNN (SGD)')

**Observation.** Adam converges faster in terms of epochs and is more forgiving with the default learning rate. SGD needs more epochs but the loss curve is smoother. Given early stopping, Adam is the more practical choice for this dataset size.

## 6. Part A.4 Ablation Study: Remove BatchNorm and Dropout

BatchNorm and Dropout are removed from the deeper architecture to directly quantify their contribution. Everything else stays the same: same 6-conv structure, same Adam optimiser, same data.

Expected outcome: faster overfitting. Training accuracy will climb higher but validation accuracy will plateau or drop earlier.

In [ ]:
ablated = build_deeper_cnn(num_classes, use_bn=False, use_dropout=False)
ablated.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

t0 = time.time()
hist_ablated = ablated.fit(
    train_ds, validation_data=val_ds,
    epochs=25, callbacks=[make_early_stopping()], verbose=1,
)
time_ablated = time.time() - t0
print(f'\nAblated model training time: {time_ablated:.1f} s')

plot_history(hist_ablated, 'Ablated CNN (no BN, no Dropout)')
y_t_abl, y_p_abl = evaluate_model(ablated, test_ds, class_names, 'Ablated CNN')

**Observation.** Without BatchNorm and Dropout the training loss goes lower but validation and test accuracy suffers. The train/val accuracy gap widens, confirming that both regularisation techniques contribute to generalisation.

## 7. Part A Summary Table

All four Part A models consolidated for direct comparison.

In [ ]:
summary_A = pd.DataFrame({
    'Model': [
        'Baseline CNN',
        'Deeper CNN (Adam)',
        'Deeper CNN (SGD)',
        'Ablated (no BN, no Dropout)',
    ],
    'Train Time (s)': [
        round(time_baseline, 1), round(time_deeper, 1),
        round(time_sgd, 1),      round(time_ablated, 1),
    ],
    'Best Val Accuracy': [
        round(max(hist_baseline.history['val_accuracy']), 4),
        round(max(hist_deeper.history['val_accuracy']),   4),
        round(max(hist_sgd.history['val_accuracy']),      4),
        round(max(hist_ablated.history['val_accuracy']),  4),
    ],
    'Total Params': [
        baseline.count_params(), deeper.count_params(),
        deeper_sgd.count_params(), ablated.count_params(),
    ],
})
print(summary_A.to_string(index=False))
summary_A

## 8. Part B Transfer Learning with MobileNetV2

**Why MobileNetV2?**
MobileNetV2 was trained on ImageNet (1.2M images, 1000 classes). Its convolutional layers have already learned to detect edges, textures, and object parts that transfer well across vision tasks. MobileNetV2 accepts inputs as small as 96x96 and its inverted residual blocks are computationally efficient.

**Two-stage strategy:**

Stage 1 (Feature Extraction): Freeze the entire pretrained backbone and train only the new classification head. The frozen backbone acts as a fixed feature extractor.

Stage 2 (Fine-Tuning): Unfreeze the top 30 backbone layers and continue training at a very low learning rate (1e-5). This lets the upper backbone layers adapt to scene-specific features while lower layers stay fixed. The small LR is critical to avoid catastrophic forgetting.

In [ ]:
IMG_SIZE_TL = (128, 128)

train_tl_full = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='training',
    seed=SEED, image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE,
    label_mode='categorical')

val_tl_full = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='validation',
    seed=SEED, image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE,
    label_mode='categorical')

vb = tf.data.experimental.cardinality(val_tl_full).numpy()
test_ds_tl = val_tl_full.take(vb // 2)
val_ds_tl  = val_tl_full.skip(vb // 2)

train_ds_tl = train_tl_full.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds_tl   = val_ds_tl.cache().prefetch(AUTOTUNE)
test_ds_tl  = test_ds_tl.cache().prefetch(AUTOTUNE)

print('TL train batches:', tf.data.experimental.cardinality(train_ds_tl).numpy())
print('TL val   batches:', tf.data.experimental.cardinality(val_ds_tl).numpy())
print('TL test  batches:', tf.data.experimental.cardinality(test_ds_tl).numpy())

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE_TL + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False
print(f'MobileNetV2: {len(base_model.layers)} layers, all frozen for Stage 1.')

data_aug_tl = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation_tl')

inputs  = keras.Input(shape=IMG_SIZE_TL + (3,))
x = data_aug_tl(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

tl_model = keras.Model(inputs, outputs, name='mobilenetv2_transfer')
tl_model.summary()

In [ ]:
tl_model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

t0 = time.time()
hist_tl = tl_model.fit(
    train_ds_tl, validation_data=val_ds_tl,
    epochs=10, callbacks=[make_early_stopping()], verbose=1,
)
time_tl = time.time() - t0
print(f'\nFeature extraction training time: {time_tl:.1f} s')

plot_history(hist_tl, 'MobileNetV2 Feature Extraction')
evaluate_model(tl_model, test_ds_tl, class_names, 'MobileNetV2 Feature Extraction')

### Stage 2: Fine-Tuning

With the classification head converged, unfreeze the top 30 backbone layers and continue with lr=1e-5. This is roughly the last two convolutional blocks of MobileNetV2, layers that encode higher-level features worth adapting.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f'Trainable backbone layers: {sum(l.trainable for l in base_model.layers)} / {len(base_model.layers)}')

tl_model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

t0 = time.time()
hist_ft = tl_model.fit(
    train_ds_tl, validation_data=val_ds_tl,
    epochs=5, callbacks=[make_early_stopping()], verbose=1,
)
time_ft = time.time() - t0
print(f'\nFine-tuning time: {time_ft:.1f} s')

plot_history(hist_ft, 'MobileNetV2 Fine-Tuning')
evaluate_model(tl_model, test_ds_tl, class_names, 'MobileNetV2 Fine-Tuned')
show_sample_predictions(tl_model, test_ds_tl, class_names)

## 9. Final Comparison: All Models

All results in one place to answer the core question: does transfer learning beat training from scratch?

In [ ]:
final_summary = pd.DataFrame({
    'Model': [
        'Baseline CNN (scratch)',
        'Deeper CNN Adam (scratch)',
        'Deeper CNN SGD (scratch)',
        'Ablated CNN (no BN/Dropout)',
        'MobileNetV2 feature extraction',
        'MobileNetV2 fine-tuned',
    ],
    'Best Val Accuracy': [
        round(max(hist_baseline.history['val_accuracy']), 4),
        round(max(hist_deeper.history['val_accuracy']),   4),
        round(max(hist_sgd.history['val_accuracy']),      4),
        round(max(hist_ablated.history['val_accuracy']),  4),
        round(max(hist_tl.history['val_accuracy']),       4),
        round(max(hist_ft.history['val_accuracy']),       4),
    ],
    'Train Time (s)': [
        round(time_baseline, 1), round(time_deeper, 1),
        round(time_sgd, 1),      round(time_ablated, 1),
        round(time_tl, 1),       round(time_tl + time_ft, 1),
    ],
})
final_summary = final_summary.sort_values('Best Val Accuracy', ascending=False).reset_index(drop=True)
print(final_summary.to_string(index=False))
final_summary

## 10. Key Takeaways

**Baseline vs Deeper:** Adding more conv layers with BatchNorm and Dropout consistently improves generalisation. The train/val gap is smaller confirming regularisation is working.

**Adam vs SGD:** Adam converges faster in fewer epochs. SGD needs more time but can reach similar performance. For this dataset size, Adam is the practical choice.

**Ablation:** Removing BatchNorm and Dropout leads to faster overfitting. Training accuracy is higher but test accuracy drops. The difference directly quantifies the regularisation benefit.

**Transfer Learning:** MobileNetV2 outperforms all from-scratch models. The pretrained ImageNet weights give it a head start that around 5000 scene images cannot compensate for.

**Limitations:** 100x100 is low resolution for scene classification. The official test folder is too sparse to be useful. Mild class imbalance may affect per-class recall for mountain images.

**Future Work:** Higher-resolution input (150x150), class weighting for imbalanced classes, test-time augmentation, EfficientNetV2 as an alternative backbone.